# 01 — Data Cleaning & Preprocessing
IPL Dataset: 1095 matches | 260,920 deliveries | Seasons 2008–2024

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

print(f'Matches: {matches.shape}')
print(f'Deliveries: {deliveries.shape}')

Matches: (1095, 20)
Deliveries: (260920, 17)


## 1. Fix Season Column

In [2]:
# Normalize seasons to single year
season_map = {
    '2007/08': '2008',
    '2009/10': '2010',
    '2020/21': '2020'
}
matches['season'] = matches['season'].replace(season_map).astype(int)
print(sorted(matches['season'].unique()))

[np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 2. Standardize Team Names

In [3]:
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Deccan Chargers': 'Deccan Chargers',  # defunct, keep as-is
    'Kochi Tuskers Kerala': 'Kochi Tuskers Kerala',  # defunct, keep as-is
    'Pune Warriors': 'Pune Warriors',  # defunct, keep as-is
    'Gujarat Lions': 'Gujarat Lions',  # defunct, keep as-is
}

for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(team_name_map)

for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_name_map)

print('Teams after normalization:', sorted(matches['team1'].unique()))

Teams after normalization: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


## 3. Handle Nulls — Matches

In [4]:
# city: fill from venue
venue_city_map = {
    'Dubai International Cricket Stadium': 'Dubai',
    'Sharjah Cricket Stadium': 'Sharjah',
    'Sheikh Zayed Stadium': 'Abu Dhabi',
    'ACA-VDCA Cricket Stadium': 'Visakhapatnam',
}
matches['city'] = matches.apply(
    lambda r: venue_city_map.get(r['venue'], r['city']) if pd.isna(r['city']) else r['city'],
    axis=1
)

# winner null = no result (rain/tie) — fill with 'No Result'
matches['winner'] = matches['winner'].fillna('No Result')
matches['player_of_match'] = matches['player_of_match'].fillna('N/A')

# method null = normal game
matches['method'] = matches['method'].fillna('Normal')

print('Nulls remaining:', matches.isnull().sum()[matches.isnull().sum() > 0])

Nulls remaining: result_margin    19
target_runs       3
target_overs      3
dtype: int64


## 4. Add Derived Columns — Matches

In [5]:
matches['date'] = pd.to_datetime(matches['date'])
matches['month'] = matches['date'].dt.month

# Did toss winner win the match?
matches['toss_winner_won'] = matches['toss_winner'] == matches['winner']

# Season-level match number
matches['match_num'] = matches.groupby('season').cumcount() + 1

print(matches[['season','date','toss_winner_won','match_num']].head())

   season       date  toss_winner_won  match_num
0    2008 2008-04-18            False          1
1    2008 2008-04-19             True          2
2    2008 2008-04-19            False          3
3    2008 2008-04-20            False          4
4    2008 2008-04-20            False          5


## 5. Handle Nulls — Deliveries

In [6]:
# These nulls are expected (most balls aren't extras or wickets)
deliveries['extras_type'] = deliveries['extras_type'].fillna('none')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('none')
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('none')
deliveries['fielder'] = deliveries['fielder'].fillna('none')

print('Deliveries nulls:', deliveries.isnull().sum().sum())

Deliveries nulls: 0


## 6. Add Derived Columns — Deliveries

In [7]:
# Phase of game
deliveries['phase'] = pd.cut(
    deliveries['over'],
    bins=[-1, 5, 14, 19],
    labels=['Powerplay (1-6)', 'Middle (7-15)', 'Death (16-20)']
)

# Is boundary?
deliveries['is_boundary'] = deliveries['batsman_runs'].isin([4, 6])
deliveries['is_four'] = deliveries['batsman_runs'] == 4
deliveries['is_six'] = deliveries['batsman_runs'] == 6
deliveries['is_dot'] = (deliveries['batsman_runs'] == 0) & (deliveries['extra_runs'] == 0)

# Merge season into deliveries
deliveries = deliveries.merge(matches[['id','season']], left_on='match_id', right_on='id', how='left')

print(deliveries[['over','phase','is_boundary','is_dot','season']].head())

   over            phase  is_boundary  is_dot  season
0     0  Powerplay (1-6)        False   False    2008
1     0  Powerplay (1-6)        False    True    2008
2     0  Powerplay (1-6)        False   False    2008
3     0  Powerplay (1-6)        False    True    2008
4     0  Powerplay (1-6)        False    True    2008


## 7. Save Cleaned Data

In [8]:
matches.to_csv('../data/processed/matches_clean.csv', index=False)
deliveries.to_csv('../data/processed/deliveries_clean.csv', index=False)

print('Saved!')
print(f'matches_clean: {matches.shape}')
print(f'deliveries_clean: {deliveries.shape}')

Saved!
matches_clean: (1095, 23)
deliveries_clean: (260920, 24)


## 8. Quick Sanity Check

In [9]:
print('=== MATCHES ===')
print(f"Seasons: {sorted(matches['season'].unique())}")
print(f"Total matches: {len(matches)}")
print(f"Unique teams: {matches['team1'].nunique()}")
print(f"Venues: {matches['venue'].nunique()}")
print()
print('=== DELIVERIES ===')
print(f"Total balls: {len(deliveries)}")
print(f"Total runs: {deliveries['total_runs'].sum()}")
print(f"Total wickets: {deliveries['is_wicket'].sum()}")
print(f"Total sixes: {deliveries['is_six'].sum()}")
print(f"Total fours: {deliveries['is_four'].sum()}")

=== MATCHES ===
Seasons: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Total matches: 1095
Unique teams: 15
Venues: 58

=== DELIVERIES ===
Total balls: 260920
Total runs: 347756
Total wickets: 12950
Total sixes: 13051
Total fours: 29850
